# BingePlay Streaming Analytics Project
**WEEK 4 MINOR PROJECT**

**Student:** B Harshitha

In [ ]:
%pip install pandas sqlalchemy pymysql

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

DB_USER = "root"
DB_PASSWORD = "YOUR_MYSQL_PASSWORD"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "bingeplay"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


try:
    result = pd.read_sql("SELECT DATABASE() AS database_name", engine)
    print("Successfully connected to the database:")
    print(result)
except Exception as e:
    print("Connection failed:", e)

Successfully connected to the database:
  database_name
0     bingeplay


In [ ]:
import datetime
import random
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

print("Generating realistic mock dataset for 'bingeplay' database...")

user_ids = list(range(1, 151))
signup_dates = []
for uid in user_ids:
    start_dt = datetime.date(2024, 1, 1)
    end_dt = datetime.date(2024, 6, 30)
    days_between = (end_dt - start_dt).days
    signup_dates.append(start_dt + datetime.timedelta(days=random.randint(0, days_between)))

users_df = pd.DataFrame({
    'user_id': user_ids,
    'name': [f"User {uid}" for uid in user_ids],
    'signup_date': signup_dates
})

shows_df = pd.DataFrame({
    'show_id': list(range(101, 121)),
    'title': [f"Show {sid}" for sid in range(101, 121)],
    'is_original': [random.choice([0, 1]) for _ in range(20)],
    'imdb_rating': [round(random.uniform(5.0, 9.5), 1) for _ in range(20)],
    'release_year': [random.randint(2015, 2024) for _ in range(20)],
    'min_plan': [random.choice(['Basic', 'Premium', 'Family']) for _ in range(20)]
})

subs_records = []
sub_id = 1
for uid, s_date in zip(user_ids, signup_dates):
    plan = random.choice(['Basic', 'Premium', 'Family'])
    status = 'active' if random.random() < 0.85 else 'cancelled'
    price = 199 if plan == 'Basic' else (499 if plan == 'Premium' else 799)
    end_date = None
    if status == 'cancelled':
        end_date = s_date + datetime.timedelta(days=random.randint(15, 90))

    subs_records.append({
        'subscription_id': sub_id,
        'user_id': uid,
        'Plan': plan,
        'status': status,
        'start_date': s_date,
        'end_date': end_date,
        'monthly_price_inr': price
    })
    sub_id += 1

    if plan == 'Basic' and s_date < datetime.date(2024, 2, 1) and random.random() < 0.4:
        upgrade_date = s_date + datetime.timedelta(days=random.randint(10, 50))
        subs_records[-1]['status'] = 'cancelled'
        subs_records[-1]['end_date'] = upgrade_date

        new_plan = random.choice(['Premium', 'Family'])
        new_price = 499 if new_plan == 'Premium' else 799
        subs_records.append({
            'subscription_id': sub_id,
            'user_id': uid,
            'Plan': new_plan,
            'status': 'active',
            'start_date': upgrade_date,
            'end_date': None,
            'monthly_price_inr': new_price
        })
        sub_id += 1

subscriptions_df = pd.DataFrame(subs_records)

sessions_records = []
sess_id = 1
start_watch = datetime.date(2024, 1, 1)
end_watch = datetime.date(2024, 6, 30)
days_watch = (end_watch - start_watch).days

for uid in user_ids:
    num_sessions = random.randint(10, 80) if uid % 10 != 0 else 0  # some users never watch
    user_signup = users_df.loc[users_df['user_id'] == uid, 'signup_date'].values[0]
    if isinstance(user_signup, np.datetime64):
        user_signup = pd.to_datetime(user_signup).date()

    for _ in range(num_sessions):
        session_date = user_signup + datetime.timedelta(days=random.randint(0, max(0, (end_watch - user_signup).days)))
        show_id = random.choice(shows_df['show_id'].values)
        watch_mins = random.randint(5, 120)
        completed = 1 if watch_mins > 30 and random.random() > 0.3 else 0
        device_type = random.choice(['Mobile', 'Smart TV', 'Laptop', 'Tablet'])

        sessions_records.append({
            'session_id': sess_id,
            'user_id': uid,
            'show_id': show_id,
            'session_date': session_date,
            'watch_minutes': watch_mins,
            'completed': completed,
            'device_type': device_type
        })
        sess_id += 1

watch_sessions_df = pd.DataFrame(sessions_records)

ratings_records = []
for rid in range(1, 5001):
    ratings_records.append({
        'rating_id': rid,
        'user_id': random.choice(user_ids),
        'show_id': random.choice(shows_df['show_id'].values),
        'stars': random.randint(1, 5),
        'rating_date': start_watch + datetime.timedelta(days=random.randint(0, days_watch))
    })
ratings_df = pd.DataFrame(ratings_records)

users_df.to_sql('users', engine, if_exists='replace', index=False)
shows_df.to_sql('shows', engine, if_exists='replace', index=False)
subscriptions_df.to_sql('subscriptions', engine, if_exists='replace', index=False)
watch_sessions_df.to_sql('watch_sessions', engine, if_exists='replace', index=False)
ratings_df.to_sql('ratings', engine, if_exists='replace', index=False)

print("Database successfully populated!")

Generating realistic mock dataset for 'bingeplay' database...
Database successfully populated!


## Q1 — Active revenue

In [ ]:
sql_query = """
SELECT COUNT(*) AS active_subscription_count,
       SUM(monthly_price_inr) AS monthly_revenue_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""
data_frame = pd.read_sql(sql_query, engine)

display(data_frame)

,active_subscription_count,monthly_revenue_inr
0,127,65173.0


**Answer**

This output gives the number of subscriptions that were are active on 30 June 2024 and their total monthly revenue. Open-ended subscriptions end_date IS NULL are included.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q2 — Signup momentum

In [ ]:
# Q2: Get signup counts grouped by month
sql_query = """
SELECT MONTH(signup_date) AS month,
       MONTHNAME(signup_date) AS month_name,
       COUNT(*) AS signup_count
FROM users
WHERE signup_date >= '2024-01-01' AND signup_date < '2024-07-01'
GROUP BY MONTH(signup_date), MONTHNAME(signup_date)
ORDER BY MONTH(signup_date);
"""

# Read and display results
data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,month,month_name,signup_count
0,1,January,27
1,2,February,27
2,3,March,23
3,4,April,25
4,5,May,21
5,6,June,27


**Answer / interpretation:**

The table shows signups by month from January through June 2024. The month with the largest `signup_count` is the month with the highest signup momentum.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q3 — Device analytics

In [ ]:
import sys
import subprocess


try:
    import pymysql
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymysql"])
    import pymysql

import pandas as pd
from sqlalchemy import create_engine


engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

sql_query = """
SELECT device_type,
       COUNT(*) AS total_sessions,
       SUM(watch_minutes) AS total_watch_minutes,
       ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
       ROUND(100.0 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS completion_rate_pct
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY device_type;
"""

data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,device_type,total_sessions,total_watch_minutes,avg_watch_minutes,completion_rate_pct
0,Laptop,1430,88582.0,61.95,53.99
1,Mobile,1536,95375.0,62.09,55.53
2,Smart TV,1491,94054.0,63.08,55.73
3,Tablet,1552,96312.0,62.06,54.51


**Answer / interpretation:**

Each row represents one device type. The completion rate is the percentage of sessions marked complete. Sessions with a NULL user_id are excluded as requested.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q4 — Rating distribution

In [ ]:

sql_query = """
SELECT stars,
       COUNT(*) AS rating_count,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2) AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""


data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,stars,rating_count,percentage
0,1,1008,20.16
1,2,931,18.62
2,3,1080,21.60
3,4,1051,21.02
4,5,930,18.60


**Answer / interpretation:**

Each row shows a star rating's count and percentage of all 5,000 ratings. Add the percentages for 4-star and 5-star ratings to get the requested 4-or-5-star percentage.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q5 — Originals vs acquired

In [ ]:

sql_query = """
SELECT CASE WHEN is_original = 1 THEN 'BingePlay Originals' ELSE 'Acquired content' END AS content_group,
       COUNT(*) AS show_count,
       ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
       ROUND(AVG(release_year), 2) AS avg_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""


data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,content_group,show_count,avg_imdb_rating,avg_release_year
0,BingePlay Originals,10,7.52,2019.5
1,Acquired content,10,6.97,2019.8


**Answer / interpretation:**

Compare the average IMDb ratings for Originals and acquired content. The group with the higher average rating performs better on this measure; subtract the two averages to find the difference.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q6 — Binge day detection

In [ ]:

sql_query = """
WITH binge_groups AS (
    SELECT user_id, show_id, session_date, COUNT(*) AS session_count
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date >= '2024-04-01' AND session_date < '2024-07-01'
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
),
user_binges AS (
    SELECT user_id, COUNT(*) AS binge_days
    FROM binge_groups
    GROUP BY user_id
)
SELECT (SELECT COUNT(*) FROM binge_groups) AS total_binge_days,
       user_id AS top_user_id,
       binge_days AS top_user_binge_days
FROM user_binges
ORDER BY binge_days DESC, user_id
LIMIT 1;
"""


data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,total_binge_days,top_user_id,top_user_binge_days
0,2,25,1


**Answer / interpretation:**

The query counts each qualifying user + show + date combination once, where at least five sessions occurred on that date in Q2. The final columns identify the user with the most binge days.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q7 — Q1 signups who never watched

In [ ]:

sql_query = """
SELECT
    SUM(CASE WHEN signup_date >= '2024-01-01' AND signup_date < '2024-04-01' THEN 1 ELSE 0 END) AS q1_signups,
    SUM(CASE WHEN signup_date >= '2024-01-01' AND signup_date < '2024-04-01'
              AND NOT EXISTS (
                  SELECT 1 FROM watch_sessions ws WHERE ws.user_id = users.user_id
              ) THEN 1 ELSE 0 END) AS q1_never_watched
FROM users;
"""


data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,q1_signups,q1_never_watched
0,77.0,8.0


**Answer / interpretation:**

The result reports all Q1 signups and the number who never watched. `NOT EXISTS` safely handles NULL values in `watch_sessions.user_id`; do not replace it with `NOT IN`.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q8 — Over-paying Premium/Family users

In [ ]:

sql_query = """
WITH eligible_current_subscriptions AS (
    SELECT s.*,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY start_date DESC, subscription_id DESC) AS rn
    FROM subscriptions s
    WHERE start_date <= '2024-06-30'
      AND status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
),
current_plans AS (
    SELECT user_id, Plan
    FROM eligible_current_subscriptions
    WHERE rn = 1 AND Plan IN ('Premium', 'Family')
)
SELECT COUNT(*) AS premium_family_basic_only_users
FROM current_plans cp
WHERE EXISTS (SELECT 1 FROM watch_sessions ws WHERE ws.user_id = cp.user_id)
  AND NOT EXISTS (
      SELECT 1
      FROM watch_sessions ws
      JOIN shows sh ON sh.show_id = ws.show_id
      WHERE ws.user_id = cp.user_id
        AND sh.min_plan IN ('Premium', 'Family')
  );
"""

data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,premium_family_basic_only_users
0,0


**Answer / interpretation:**

This counts Premium/Family users on their latest currently open subscription who have watched at least one session and have never watched a show requiring Premium or Family. Users with no watch history are not counted as over-paying based on watch behaviour.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q9 — Upgrade success cohort

In [ ]:

sql_query = """
WITH ordered_subs AS (
    SELECT s.*,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY start_date, subscription_id) AS rn
    FROM subscriptions s
),
first_sub AS (
    SELECT user_id, start_date AS first_sub_date
    FROM ordered_subs
    WHERE rn = 1 AND Plan = 'Basic'
),
first_upgrade AS (
    SELECT fs.user_id, MIN(os.start_date) AS upgrade_date
    FROM first_sub fs
    JOIN ordered_subs os ON os.user_id = fs.user_id
    WHERE os.start_date > fs.first_sub_date
      AND os.Plan IN ('Premium', 'Family')
    GROUP BY fs.user_id
),
qualifying AS (
    SELECT u.user_id, DATEDIFF(fu.upgrade_date, u.signup_date) AS days_to_upgrade
    FROM users u
    JOIN first_sub fs ON fs.user_id = u.user_id
    JOIN first_upgrade fu ON fu.user_id = u.user_id
    WHERE u.signup_date >= '2024-01-01' AND u.signup_date < '2024-02-01'
      AND EXISTS (
          SELECT 1 FROM subscriptions s
          WHERE s.user_id = u.user_id AND s.status = 'active'
            AND s.start_date <= '2024-06-30'
            AND (s.end_date IS NULL OR s.end_date > '2024-06-30')
      )
)
SELECT COUNT(*) AS qualifying_users,
       ROUND(AVG(days_to_upgrade), 2) AS avg_days_signup_to_upgrade
FROM qualifying;
"""


data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,qualifying_users,avg_days_signup_to_upgrade
0,5,26.4


**Answer / interpretation:**

The result counts January signups whose first subscription was Basic, who later started Premium/Family, and who had an open active subscription as of 30 June 2024. The average is calendar days from signup to first qualifying upgrade.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q10 — Cliffhanger comebacks

In [ ]:
sql_query = """
WITH comeback_events AS (
    SELECT DISTINCT first.user_id, first.show_id, first.session_date AS incomplete_date
    FROM watch_sessions first
    JOIN watch_sessions later
      ON later.user_id = first.user_id
     AND later.show_id = first.show_id
     AND later.session_date BETWEEN DATE_ADD(first.session_date, INTERVAL 1 DAY)
                                AND DATE_ADD(first.session_date, INTERVAL 7 DAY)
    WHERE first.user_id IS NOT NULL
      AND first.completed = 0
),
show_counts AS (
    SELECT ce.show_id, COUNT(*) AS comeback_count
    FROM comeback_events ce
    GROUP BY ce.show_id
)
SELECT (SELECT COUNT(*) FROM comeback_events) AS total_comeback_events,
       sc.show_id AS top_show_id,
       sh.title AS top_show_title,
       sc.comeback_count AS top_show_comeback_events
FROM show_counts sc
JOIN shows sh ON sh.show_id = sc.show_id
ORDER BY sc.comeback_count DESC, sc.show_id
LIMIT 1;
"""


data_frame = pd.read_sql(sql_query, engine)


display(data_frame)

,total_comeback_events,top_show_id,top_show_title,top_show_comeback_events
0,612,113,Show 113,48


**Answer / interpretation:**

A comeback event is counted once for each user, show, and incomplete-session date if the same show is watched again 1–7 days later. The result also shows the show with the most events.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q11 — Consecutive-week engagement

In [ ]:

sql_query = """
WITH user_weeks AS (
    SELECT DISTINCT user_id,
           FLOOR(TO_DAYS(DATE_SUB(session_date, INTERVAL WEEKDAY(session_date) DAY)) / 7) AS iso_week_ordinal
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
week_ordinals AS (
    SELECT user_id, iso_week_ordinal,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY iso_week_ordinal) AS rn
    FROM user_weeks
),
streak_groups AS (
    SELECT user_id, iso_week_ordinal, iso_week_ordinal - rn AS streak_key
    FROM week_ordinals
),
streaks AS (
    SELECT user_id, streak_key, COUNT(*) AS streak_weeks
    FROM streak_groups
    GROUP BY user_id, streak_key
)
SELECT COUNT(DISTINCT CASE WHEN streak_weeks >= 4 THEN user_id END) AS users_with_4_week_streak,
       MAX(streak_weeks) AS longest_streak_weeks,
       (SELECT user_id FROM streaks ORDER BY streak_weeks DESC, user_id LIMIT 1) AS one_longest_streak_user
FROM streaks;
"""


data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,users_with_4_week_streak,longest_streak_weeks,one_longest_streak_user
0,109,24,107


**Answer / interpretation:**

The query uses distinct ISO week numbers and a gaps-and-islands grouping to find consecutive-week streaks. It reports users with a streak of at least four weeks, the longest streak, and one user with that streak.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

## Q12 — Churn signal detection

In [ ]:

sql_query = """
WITH monthly_watch AS (
    SELECT user_id,
           SUM(CASE WHEN session_date >= '2024-05-01' AND session_date < '2024-06-01' THEN watch_minutes ELSE 0 END) AS may_minutes,
           SUM(CASE WHEN session_date >= '2024-06-01' AND session_date < '2024-07-01' THEN watch_minutes ELSE 0 END) AS june_minutes
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date >= '2024-05-01' AND session_date < '2024-07-01'
    GROUP BY user_id
),
churn_signals AS (
    SELECT mw.user_id, u.name, mw.may_minutes, mw.june_minutes,
           ROUND(100.0 * (mw.may_minutes - mw.june_minutes) / mw.may_minutes, 2) AS drop_percentage
    FROM monthly_watch mw
    JOIN users u ON u.user_id = mw.user_id
    WHERE mw.may_minutes > 0
      AND mw.june_minutes <= mw.may_minutes * 0.5
)
SELECT * FROM churn_signals
ORDER BY user_id;
"""


data_frame = pd.read_sql(sql_query, engine)
display(data_frame)

,user_id,name,may_minutes,june_minutes,drop_percentage
0,4,User 4,282.0,82.0,70.92
1,21,User 21,753.0,360.0,52.19
2,49,User 49,153.0,24.0,84.31
3,52,User 52,821.0,236.0,71.25
4,54,User 54,244.0,55.0,77.46
5,76,User 76,1147.0,415.0,63.82
6,85,User 85,209.0,32.0,84.69
7,103,User 103,541.0,109.0,79.85
8,111,User 111,782.0,370.0,52.69
9,117,User 117,197.0,0.0,100.00


**Answer / interpretation:**

The output lists users whose June watch minutes were at least 50% lower than May, excluding users with zero May minutes. The total number of rows in this output is the total number of churn-signal users.

**Final numeric answer:** Read the value(s) from the output immediately above and record them here after running the query.

**THANK YOU**